# Phi-3 · Particiones y SFT

En VS Code selecciona **rocm (3.14.10)** o **Tarea 3 (GPU)**. Para instalarlo o ejecutar en Colab, sigue el [README](../README.md#entorno).

Construimos las cuatro particiones de HH-RLHF con semilla 42. Para SFT usamos la respuesta preferida y enmascaramos el prompt y el padding con `-100`. El modelo base queda congelado en 4 bits y los adaptadores LoRA se entrenan durante una época.

Ejecuta las primeras tres celdas de código para obtener las particiones. Para el piloto, fija `PILOTO = True` antes de entrenar. La corrida definitiva usa `PILOTO = False`.

In [ ]:
from pathlib import Path
import sys

folder = Path.cwd() if Path("comun.py").is_file() else Path.cwd() / "tarea3"
if not (folder / "comun.py").is_file():
    raise FileNotFoundError("Abre el cuaderno desde la carpeta tarea3 o la raiz del repositorio.")
sys.path.insert(0, str(folder))

from comun import preparar_notebook

preparar_notebook()

In [ ]:
import json
import math
import os

from comun import MODEL_REV, OUT, SEED, SFT_DIR, SFT_LR, load_model, load_tokenizer, prepare_partitions, sft_example

PILOTO = os.getenv("TAREA3_PILOTO") == "1"
print("Cargando tokenizador de Phi-3…", flush=True)
tokenizer = load_tokenizer()

In [ ]:
print("Preparando las cuatro particiones de HH-RLHF…", flush=True)
folder = prepare_partitions(tokenizer)
for path in sorted(folder.glob("*.jsonl")):
    print(path.name, len(path.read_text(encoding="utf-8").splitlines()))

## Inspección de las respuestas preferidas

Una respuesta `chosen` puede facilitar daño. Revisar los dos ejemplos de cada fuente ayuda a reconocer esa limitación antes de interpretar los resultados.

In [ ]:
rows = [json.loads(line) for line in (folder / "sft.jsonl").read_text(encoding="utf-8").splitlines()]
for source in ("helpful-base", "harmless-base"):
    sample = [row for row in rows if row["source"] == source][:2]
    for row in sample:
        print(row["id"], "\nSolicitud:", row["prompt"], "\nElegida:", row["chosen"], "\n")

## Entrenamiento QLoRA

La variante usa una tasa de aprendizaje de 5e-5. La corrida completa se guarda en `salidas/sft_lr5e5` y el piloto de un paso en `salidas/sft_lr5e5_piloto`. El adaptador de referencia está en `salidas/sft`.

In [ ]:
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments, set_seed

set_seed(SEED)
selected = rows[:4] if PILOTO else rows
examples = [sft_example(tokenizer, row["prompt"], row["chosen"]) for row in selected]
assert len(rows) == 512 and all(example is not None for example in examples)
for example in examples:
    first_target = next(i for i, label in enumerate(example["labels"]) if label != -100)
    assert first_target > 0 and example["labels"] == (
        [-100] * first_target + example["input_ids"][first_target:]
    )
collator = DataCollatorForSeq2Seq(tokenizer, label_pad_token_id=-100, pad_to_multiple_of=8)
batch = collator(examples[:2])
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)

print("Cargando Phi-3 en la GPU…", flush=True)
model = load_model()
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={"use_reentrant": False},
)
model = get_peft_model(
    model,
    LoraConfig(
        task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
        target_modules=["qkv_proj", "o_proj"],
    ),
    revision=MODEL_REV,
)
model.print_trainable_parameters()
adapter_weight = next(param for name, param in model.named_parameters() if "lora_B." in name)
assert adapter_weight.requires_grad
before = adapter_weight.detach().cpu().clone()
print(torch.cuda.get_device_name(0), "ROCm" if torch.version.hip else "CUDA")

In [ ]:
output_dir = OUT / (f"{SFT_DIR}_piloto" if PILOTO else SFT_DIR)
args = TrainingArguments(
    output_dir=str(output_dir / "checkpoints"),
    seed=SEED,
    num_train_epochs=1,
    max_steps=1 if PILOTO else -1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=SFT_LR,
    fp16=True,
    activation_checkpointing=True,
    activation_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=1 if PILOTO else 5,
    disable_tqdm=False,
    save_strategy="no" if PILOTO else "steps",
    save_steps=32,
    save_total_limit=2,
    report_to="none",
)
trainer = Trainer(
    model=model, args=args, train_dataset=examples, data_collator=collator, processing_class=tokenizer,
)
print(f"Iniciando SFT con {len(examples)} ejemplos ({'piloto' if PILOTO else 'corrida final'})…", flush=True)
result = trainer.train()
assert math.isfinite(result.training_loss) and not torch.equal(before, adapter_weight.detach().cpu())
trainer.save_model(str(output_dir))
trainer.state.save_to_json(str(output_dir / "trainer_state.json"))
trainer.save_metrics("train", result.metrics)
print("Adaptador SFT:", output_dir)